## Structured Output

MOdel can be requested to provide their response in a formate matching a given Schema. This is usefull for ensuring the output can be easily parsed and used in subsequent processing. LangChain support multiple schema types and method for enforcing structured output.


### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables from .env
load_dotenv(find_dotenv(), override=True)

# Initialize Groq chat model
model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500)
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x0000027FCCDD0220>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027FCCDD0130>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=500)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year the movie was released")
    director: str=Field(description="The Director of the movie")
    rating: float=Field(description="The movie rating out of 10")
    


In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x0000027FCCDD0220>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027FCCDD0130>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=500), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The Director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function'

In [4]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='**Inception** is a 2010 science fiction action film written and directed by **Christopher Nolan**. It is widely regarded as one of the most complex and influential films of the 21st century, known for its intricate plot, visual effects, and philosophical themes.\n\n### 📽️ Basic Information\n- **Title:** Inception\n- **Release Date:** July 16, 2010 (US)\n- **Director:** Christopher Nolan\n- **Screenwriter:** Christopher Nolan\n- **Producers:** Emma Thomas, Christopher Nolan\n- **Music:** Hans Zimmer\n- **Runtime:** 148 minutes\n- **Genre:** Science Fiction, Action, Thriller\n- **Studio:** Legendary Pictures, Syncopy\n- **Distributor:** Warner Bros. Pictures\n\n### 🎭 Main Cast\n- **Leonardo DiCaprio** as **Dom Cobb** – A skilled thief who specializes in "extraction," stealing secrets from deep within the subconscious during the dream state.\n- **Joseph Gordon-Levitt** as **Arthur** – Cobb’s point man and planner.\n- **Elliot Page** (credited as Ellen Page) as **Ariadne

In [6]:
model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year the movie was released")
    director: str=Field(description="The Director of the movie")
    rating: float=Field(description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response=model_with_structure.invoke("Provide the details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'menxvpdxv', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 350, 'total_tokens': 418, 'completion_time': 0.170274222, 'completion_tokens_details': None, 'prompt_time': 0.023918682, 'prompt_tokens_details': None, 'queue_time': 0.050357837, 'total_time': 0.194192904}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e667-1e0f-7fb1-a662-24e206a58c6a-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'menxvpdxv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 350, 'output_tokens': 68, 'total

### Nested Structure

In [8]:
class Actor(BaseModel):
    name:str
    role:str

class Moviedetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budge: float | None= Field(None, description="Budget in million USD")

In [9]:

model_with_structure = model.with_structured_output(Moviedetails, include_raw=True)
response=model_with_structure.invoke("Provide the details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kgb5qe1hj', 'function': {'arguments': '{"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Elliot Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Marion Cotillard","role":"Mal Cobb"},{"name":"Cillian Murphy","role":"Robert Fischer"}],"genres":["Action","Science Fiction","Thriller"],"title":"Inception","year":2010}', 'name': 'Moviedetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 392, 'total_tokens': 581, 'completion_time': 0.38963951, 'completion_tokens_details': None, 'prompt_time': 0.027558557, 'prompt_tokens_details': None, 'queue_time': 0.049756922, 'total_time': 0.417198067}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

### TypedDict

TypeDict provide a simpler alternative using python's built-in typing, ideal when you don't need runtime validation.

In [10]:
from typing_extensions import TypedDict, Annotated

In [11]:
class MovieDict(TypedDict):
    """A Movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the Movie was relesed "]
    director: Annotated[str, ..., "The Director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [12]:
model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [14]:
class Actor(TypedDict):
    name:str
    role:str

class Moviedetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None= Field(None, description="Budget in million USD")

model_withtypedict=model.with_structured_output(Moviedetails)
response=model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / The Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [15]:
model.profile

## Data Classes

A data class is a class typically containing mainly data, although there aren't really any reactions. You create it using the @dataclass decorator.

In [ ]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =create_agent(
    model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500),
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='33fa232c-1673-4f6e-a363-dff4dbbe086b'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pkfet6zs5', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 351, 'total_tokens': 419, 'completion_time': 0.150694141, 'completion_tokens_details': None, 'prompt_time': 0.0249989, 'prompt_tokens_details': None, 'queue_time': 0.134924669, 'total_time': 0.175693041}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e667-d914-7a03-814c-70c2307661cc-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@examp

In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [21]:
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =create_agent(
    model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500),
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [22]:
# Dataclass

from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =create_agent(
    model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500),
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')